# Neural networks and the training loop

Machine Learning Engineer track, lesson MLE5. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** describe a neuron, a layer and a loss, write the training loop of forward pass, loss, backward pass and update, see what the learning rate does, and read a training and validation loss curve that overfits.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd
import torch
from torch import nn

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole track
torch.set_num_threads(1)       # one thread: the sums are added in the same order on every run
DEVICE = "cpu"                 # this network is small: a GPU would not make it faster (the GPU lessons come later)

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split, the scaled columns and the tensors

The split and the fitting and validation parts of the Data Scientist track (10,021 and 3,341 people; the test part is not scored in this lesson). The text columns become 0 or 1 columns and the numbers are scaled, with what the fitting people say, as in lesson MLE1: 19 columns in all. A *tensor* is PyTorch's table of numbers. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)
prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), make_column_selector(dtype_exclude="number")),
                          ("num", StandardScaler(), make_column_selector(dtype_include="number"))]).fit(X_fit)


def tensors(rows, labels):
    """The scaled columns and the 0 or 1 labels as float tensors; the labels are a column of numbers."""
    return torch.tensor(prep.transform(rows), dtype=torch.float32), torch.tensor(labels.to_numpy(), dtype=torch.float32).unsqueeze(1)


Xf, yf = tensors(X_fit, y_fit)
Xv, yv = tensors(X_val, y_val)
logit = LogisticRegression(max_iter=1000).fit(prep.transform(X_fit), y_fit)       # the day-7 model, on the same columns
logit_chances = logit.predict_proba(prep.transform(X_val))[:, 1]
print(Xf.shape[0], Xv.shape[0], Xf.shape[1], round(float(roc_auc_score(y_val, logit_chances)), 3))

## Unit 1: A neuron, a layer, and their numbers

**Predict first.** A single neuron reads the 19 columns and gives one chance; a small network has a layer of 8 neurons, then one output neuron. How many learnable numbers does each have?

In [ ]:
neuron = nn.Sequential(nn.Linear(19, 1), nn.Sigmoid())                     # one neuron: 19 weights, one bias, a bend into 0 to 1
small = nn.Sequential(nn.Linear(19, 8), nn.ReLU(), nn.Linear(8, 1))        # a layer of 8 neurons, a bend (ReLU), one output neuron


def count(model):
    """How many numbers the model learns."""
    return sum(parameter.numel() for parameter in model.parameters())


print(count(neuron), count(small))
with torch.no_grad():                                                      # give the neuron the logistic regression's own numbers
    neuron[0].weight.copy_(torch.tensor(logit.coef_, dtype=torch.float32))
    neuron[0].bias.copy_(torch.tensor(logit.intercept_, dtype=torch.float32))
    print(round(float((neuron(Xv).squeeze(1) - torch.tensor(logit_chances, dtype=torch.float32)).abs().max()), 6))

## Unit 2: A loss says how wrong the chances are

**Predict first.** Binary cross-entropy gives a person who churned the loss -ln(chance) and a person who stayed -ln(1 - chance), averaged over the people. What loss will a chance of 0.5 for everyone give on the fitting people, what will the share of churners (about 0.54) give, and what will a new, untrained network give?

In [ ]:
loss_fn = nn.BCELoss()                                                     # binary cross-entropy
torch.manual_seed(SEED)
untrained = nn.Sequential(nn.Linear(19, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
share = float(yf.mean())
with torch.no_grad():
    print(round(float(loss_fn(torch.full_like(yf, 0.5), yf)), 3), round(float(loss_fn(torch.full_like(yf, share), yf)), 3), round(float(loss_fn(untrained(Xf), yf)), 3))

## Unit 3: The training loop, and the size of its step

**Predict first.** The loop below trains the 19, 8, 1 network for 20 epochs (an epoch is one pass over the fitting people, in batches of 256) with plain gradient descent. Tried with learning rates 0.01, 0.1, 1.0 and 10.0, which will end with the lowest loss on the fitting people?

In [ ]:
def fit_network(Xa, ya, hidden, lr, epochs, batch=256, seed=SEED):
    """Train a network with one hidden layer; return it and the loss on the fitting and validation people after each epoch."""
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(Xa.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, 1), nn.Sigmoid())
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)                 # the update: every number moves by lr times its slope
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(Xa))                                    # a new shuffle of the people every epoch
        for start in range(0, len(Xa), batch):
            rows = order[start:start + batch]
            optimizer.zero_grad()                                          # forget the slopes of the last batch
            loss = loss_fn(model(Xa[rows]), ya[rows])                      # forward pass, then the loss of this batch
            loss.backward()                                                # backward pass: the slope of the loss for every number
            optimizer.step()                                               # update
        with torch.no_grad():
            history.append((float(loss_fn(model(Xa), ya)), float(loss_fn(model(Xv), yv))))
    return model, history


for lr in [0.01, 0.1, 1.0, 10.0]:
    model, history = fit_network(Xf, yf, 8, lr, 20)
    with torch.no_grad():
        auc = roc_auc_score(y_val, model(Xv).squeeze(1).numpy())
    print(lr, round(history[-1][0], 3), round(history[-1][1], 3), round(float(auc), 3))

## Unit 4: Two curves, and where they part

**Predict first.** A bigger network, one hidden layer of 64 neurons, learns from only 500 of the fitting people for 300 epochs (batches of 50, learning rate 0.1). The loss on those 500 and on the 3,341 validation people is recorded after every epoch. At about which epoch will the validation loss be lowest, and will it be higher or lower at the end than at its lowest?

In [ ]:
pilot = X_fit.sample(500, random_state=SEED)                               # 500 of the fitting people
Xp, yp = tensors(pilot, y_fit.loc[pilot.index])
big, curves = fit_network(Xp, yp, 64, 0.1, 300, batch=50)
validation = [pair[1] for pair in curves]
best_epoch = int(np.argmin(validation)) + 1
print(best_epoch, round(validation[best_epoch - 1], 3))
print(round(curves[-1][0], 3), round(curves[-1][1], 3))

A picture of the same two curves, for Colab. Run this cell as it is.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 301), [pair[0] for pair in curves], label="loss on the 500 people it learns from")
plt.plot(range(1, 301), validation, label="loss on the validation people")
plt.axvline(best_epoch, color="gray", linestyle=":")
plt.xlabel("epoch")
plt.ylabel("binary cross-entropy")
plt.legend()
plt.show()

## Finish it

### Exercise 1: Finish it

**Task:** **Count a deeper network.** A network has two hidden layers: 19 columns in, 16 neurons, then 8 neurons, then one output neuron, with a ReLU after each hidden layer. Complete the model and put its number of learnable numbers in `answer_1`.

**Expected:** `answer_1` is a whole number.

In [ ]:
deeper = nn.Sequential(nn.Linear(19, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU())      # given: the two hidden layers
# add the output neuron to the model, then count its learnable numbers with count()
answer_1 = None  # replace None with the number

In [ ]:
_ = check("Exercise 1", "answer_1", 465, traps={456: "That leaves out the bias of the 8-neuron layer or the output neuron: every neuron has a bias."})

<details>
<summary>Hint</summary>

The output neuron reads the 8 outputs of the layer before it: `nn.Linear(8, 1)`. Add it with `deeper.append(...)` or build the whole `nn.Sequential` again, then call `count`.

</details>

<details>
<summary>Half-solution</summary>

```python
deeper = nn.Sequential(nn.Linear(19, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(..., 1))
answer_1 = count(deeper)
```

</details>

<details>
<summary>Solution</summary>

```python
deeper = nn.Sequential(nn.Linear(19, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
answer_1 = count(deeper)
```

465 numbers: (19 + 1) x 16 = 320, (16 + 1) x 8 = 136 and (8 + 1) x 1 = 9, where each neuron has one weight for every input it reads and one bias.

</details>

## On your own

### Exercise 2: On your own

**Task:** **The loop from a blank cell.** Before you start, say in one sentence what the loss does as the steps go on: it falls, by less each time. Write the training loop yourself, without `fit_network`: a network `net` with 8 hidden neurons (`nn.Sequential(nn.Linear(19, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())`, built after `torch.manual_seed(SEED)`), gradient descent with learning rate 0.1, and 100 steps that each use all 10,021 fitting people at once. Put the loss on the fitting people after step 100, rounded to 3 decimals, in `answer_2`. Write the code in the empty cell of the notebook.

**Expected:** `answer_2` is a number a little below 0.69.

In [ ]:
_ = check("Exercise 2", "answer_2", 0.666, tol=5e-4)

<details>
<summary>Hint</summary>

Each step is the same five lines as in unit 3 without the batch loop: `zero_grad`, the loss of `net(Xf)` against `yf`, `backward`, `step`. After the 100 steps compute the loss once more under `torch.no_grad()`.

</details>

<details>
<summary>Half-solution</summary>

```python
torch.manual_seed(SEED)
net = nn.Sequential(nn.Linear(19, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
for step in range(100):
    optimizer.zero_grad()
    ...
```

</details>

<details>
<summary>Solution</summary>

```python
torch.manual_seed(SEED)
net = nn.Sequential(nn.Linear(19, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
for step in range(100):
    optimizer.zero_grad()
    loss = loss_fn(net(Xf), yf)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    answer_2 = round(float(loss_fn(net(Xf), yf)), 3)
```

The loss after 100 full-batch steps is 0.666: below the 0.693 of a chance of 0.5 for everyone, and well above the 0.631 that unit 3 reached after 20 epochs of 40 steps each. A step on all the people moves the numbers by the average slope, and 100 of them are fewer updates than the 800 of unit 3.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague copies the loop of unit 3 and records the loss of an epoch as the loss of its last batch, `float(loss)`. Change the function so that it records the loss on all the fitting people after each epoch, and put the lowest of the 20 epoch losses (8 neurons, learning rate 0.1), rounded to 3 decimals, in `answer_3`.

**Expected:** `answer_3` is a number between 0.5 and 0.8.

In [ ]:
def fit_last_batch(Xa, ya, hidden, lr, epochs, batch=256, seed=SEED):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(Xa.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, 1), nn.Sigmoid())
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(Xa))
        for start in range(0, len(Xa), batch):
            rows = order[start:start + batch]
            optimizer.zero_grad()
            loss = loss_fn(model(Xa[rows]), ya[rows])
            loss.backward()
            optimizer.step()
        history.append(float(loss))                                        # colleague: "the loss of the epoch is the loss of its last batch"
    return history


answer_3 = round(min(fit_last_batch(Xf, yf, 8, 0.1, 20)), 3)
# change the function so that it records the loss on all the fitting people after each epoch

In [ ]:
_ = check("Exercise 3", "answer_3", 0.631, traps={0.543: "That is the lowest loss of a last batch, a few dozen people. Measure all the fitting people."}, tol=5e-4)

<details>
<summary>Hint</summary>

The model sees different people in every batch. How many people are in the last batch of an epoch of 10,021 people in batches of 256? Where in the loop can you measure all of them under `torch.no_grad()`?

</details>

<details>
<summary>Half-solution</summary>

```python
        with torch.no_grad():
            history.append(float(loss_fn(model(Xa), ya)))
```

</details>

<details>
<summary>Solution</summary>

```python
def fit_last_batch(Xa, ya, hidden, lr, epochs, batch=256, seed=SEED):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(Xa.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, 1), nn.Sigmoid())
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(Xa))
        for start in range(0, len(Xa), batch):
            rows = order[start:start + batch]
            optimizer.zero_grad()
            loss = loss_fn(model(Xa[rows]), ya[rows])
            loss.backward()
            optimizer.step()
        with torch.no_grad():
            history.append(float(loss_fn(model(Xa), ya)))
    return history


answer_3 = round(min(fit_last_batch(Xf, yf, 8, 0.1, 20)), 3)
```

The lowest of the 20 losses is 0.631, the loss after the last epoch. The flawed function reports 0.543, because the last batch of each epoch holds only 37 people (10,021 = 39 x 256 + 37), and the lowest of 20 noisy values is a lucky one. A loss curve is measured on the same people every time.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")